# 窃电检测机器学习流程

本 Notebook 演示基于 SGCC 数据集的窃电检测完整工作流，包括：

1. 数据加载
2. 数据探索与可视化
3. 特征工程
4. 训练 / 验证 / 测试集划分
5. 基线模型训练（随机森林 + XGBoost）
6. 模型评估（准确率、精确率、召回率、F1、ROC-AUC、PR-AUC）

> **数据说明**：请将 `data.csv`（用电时序数据）与 `label.csv`（用户标签）放置在本仓库根目录的 `data/` 文件夹中。
> 数据集来源参考：[SGCC Dataset](https://github.com/henryRDlab/ElectricityTheftDetection)

## 0. 导入依赖库

In [ ]:
import os
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score,
    roc_curve, precision_recall_curve, confusion_matrix, ConfusionMatrixDisplay
)
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE
import xgboost as xgb

# 设置随机种子，保证结果可复现
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# 设置中文字体（Codespaces 环境下使用英文标签，避免字体缺失问题）
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['axes.unicode_minus'] = False

print('依赖库导入完成')

## 1. 数据加载

使用相对路径从 `../data/` 读取数据，确保在不同环境下均可运行。

In [ ]:
# 数据路径（相对于本 Notebook 所在目录）
DATA_DIR = os.path.join('..', 'data')
DATA_PATH  = os.path.join(DATA_DIR, 'data.csv')
LABEL_PATH = os.path.join(DATA_DIR, 'label.csv')

# 检查数据文件是否存在
if not os.path.exists(DATA_PATH) or not os.path.exists(LABEL_PATH):
    print('⚠  未找到数据文件，将使用随机生成的示例数据演示流程。')
    print(f'   预期路径：{os.path.abspath(DATA_PATH)}  /  {os.path.abspath(LABEL_PATH)}')
    USE_DEMO = True
else:
    USE_DEMO = False

if USE_DEMO:
    # ---- 生成示例数据（仅用于演示，不代表真实场景） ----
    N_USERS = 500
    N_DAYS  = 1035   # 与 SGCC 数据集列数对齐
    rng = np.random.default_rng(RANDOM_STATE)

    normal_data = rng.normal(loc=10, scale=3, size=(int(N_USERS * 0.9), N_DAYS)).clip(0)
    theft_data  = rng.normal(loc=5,  scale=5, size=(int(N_USERS * 0.1), N_DAYS)).clip(0)
    raw_data = np.vstack([normal_data, theft_data])

    # 随机插入缺失值
    mask = rng.random(raw_data.shape) < 0.05
    raw_data[mask] = np.nan

    date_cols = [f'CONS_MM_DD_{i:04d}' for i in range(N_DAYS)]
    df_data  = pd.DataFrame(raw_data, columns=date_cols)
    df_data.insert(0, 'CONS_NO', [f'U{i:05d}' for i in range(N_USERS)])

    labels = np.array([0] * int(N_USERS * 0.9) + [1] * int(N_USERS * 0.1))
    df_label = pd.DataFrame({'CONS_NO': df_data['CONS_NO'], 'FLAG': labels})
else:
    # ---- 读取真实数据 ----
    df_data  = pd.read_csv(DATA_PATH)
    df_label = pd.read_csv(LABEL_PATH)

print(f'用电数据维度：{df_data.shape}')
print(f'标签数据维度：{df_label.shape}')
df_data.head(3)

## 2. 数据探索与可视化

In [ ]:
# 2.1 标签分布
label_counts = df_label['FLAG'].value_counts().sort_index()
print('标签分布：')
print(label_counts.rename({0: '正常用户(0)', 1: '窃电用户(1)'}))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].bar(['Normal(0)', 'Theft(1)'], label_counts.values, color=['steelblue', 'tomato'])
axes[0].set_title('User Label Distribution')
axes[0].set_ylabel('Count')
for i, v in enumerate(label_counts.values):
    axes[0].text(i, v + 1, str(v), ha='center')

axes[1].pie(
    label_counts.values,
    labels=['Normal(0)', 'Theft(1)'],
    autopct='%1.1f%%',
    colors=['steelblue', 'tomato'],
    startangle=90
)
axes[1].set_title('Label Proportion')
plt.tight_layout()
plt.show()

In [ ]:
# 2.2 缺失值分析
feature_cols = [c for c in df_data.columns if c != 'CONS_NO']
missing_rate = df_data[feature_cols].isnull().mean()
print(f'时序列总数：{len(feature_cols)}')
print(f'含缺失值的列数：{(missing_rate > 0).sum()}')
print(f'整体缺失率：{missing_rate.mean():.2%}')

plt.figure(figsize=(12, 3))
plt.plot(missing_rate.values, linewidth=0.8, color='darkorange')
plt.xlabel('Day Index')
plt.ylabel('Missing Rate')
plt.title('Missing Rate per Day')
plt.tight_layout()
plt.show()

In [ ]:
# 2.3 正常用户 vs 窃电用户平均用电曲线对比
df_merged = df_data.merge(df_label, on='CONS_NO')
normal_mean = df_merged[df_merged['FLAG'] == 0][feature_cols].mean()
theft_mean  = df_merged[df_merged['FLAG'] == 1][feature_cols].mean()

plt.figure(figsize=(14, 4))
plt.plot(normal_mean.values, label='Normal User', alpha=0.8, linewidth=1)
plt.plot(theft_mean.values,  label='Theft User',  alpha=0.8, linewidth=1, color='tomato')
plt.xlabel('Day Index')
plt.ylabel('Average Electricity Usage')
plt.title('Average Daily Usage: Normal vs Theft')
plt.legend()
plt.tight_layout()
plt.show()

## 3. 特征工程

从时序用电数据中提取统计特征，包括：
- 均值、标准差、最大值、最小值、中位数
- 四分位距（IQR）
- 偏度（Skewness）、峰度（Kurtosis）
- 一阶差分均值与标准差（反映用电变化幅度）
- 缺失值比例
- 非零比例

In [ ]:
def extract_features(df: pd.DataFrame, feature_cols: list) -> pd.DataFrame:
    """从时序用电数据中提取统计特征。"""
    vals = df[feature_cols].values.astype(float)
    diff_vals = np.diff(vals, axis=1)

    feat = pd.DataFrame(index=df.index)
    feat['mean']        = np.nanmean(vals, axis=1)
    feat['std']         = np.nanstd(vals,  axis=1)
    feat['max']         = np.nanmax(vals,  axis=1)
    feat['min']         = np.nanmin(vals,  axis=1)
    feat['median']      = np.nanmedian(vals, axis=1)
    feat['q75']         = np.nanpercentile(vals, 75, axis=1)
    feat['q25']         = np.nanpercentile(vals, 25, axis=1)
    feat['iqr']         = feat['q75'] - feat['q25']

    # 偏度与峰度（忽略全 NaN 行）
    df_tmp = pd.DataFrame(vals)
    feat['skew']        = df_tmp.skew(axis=1).values
    feat['kurt']        = df_tmp.kurt(axis=1).values

    # 差分特征
    feat['diff_mean']   = np.nanmean(np.abs(diff_vals), axis=1)
    feat['diff_std']    = np.nanstd(diff_vals,  axis=1)

    # 缺失值比例
    feat['missing_rate'] = np.isnan(vals).mean(axis=1)

    # 非零比例
    feat['nonzero_rate'] = np.nanmean(vals > 0, axis=1)

    feat['CONS_NO'] = df['CONS_NO'].values
    return feat


df_features = extract_features(df_data, feature_cols)
print(f'特征维度：{df_features.shape}')
df_features.head(3)

In [ ]:
# 合并特征与标签
df_final = df_features.merge(df_label, on='CONS_NO')

feat_names = [c for c in df_final.columns if c not in ('CONS_NO', 'FLAG')]
X = df_final[feat_names].values
y = df_final['FLAG'].values

# 用均值填充 NaN（极端情况下某统计量可能为 NaN）
col_means = np.nanmean(X, axis=0)
nan_mask  = np.isnan(X)
X[nan_mask] = np.take(col_means, np.where(nan_mask)[1])

print(f'最终特征矩阵：{X.shape}，标签：{y.shape}')
print(f'窃电用户数量：{y.sum()}，正常用户：{(y == 0).sum()}')

## 4. 数据集划分

采用分层采样，按 **70% 训练 / 15% 验证 / 15% 测试** 的比例划分。

In [ ]:
# 先划分出测试集（15%）
X_trainval, X_test, y_trainval, y_test = train_test_split(
    X, y, test_size=0.15, random_state=RANDOM_STATE, stratify=y
)

# 再从剩余数据中划分验证集（约 15% / 85% ≈ 17.6%）
X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval,
    test_size=0.15 / 0.85,
    random_state=RANDOM_STATE,
    stratify=y_trainval
)

print(f'训练集：{X_train.shape}，窃电比例：{y_train.mean():.2%}')
print(f'验证集：{X_val.shape}，  窃电比例：{y_val.mean():.2%}')
print(f'测试集：{X_test.shape}，  窃电比例：{y_test.mean():.2%}')

In [ ]:
# SMOTE 过采样（仅对训练集）
smote = SMOTE(random_state=RANDOM_STATE)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)
print(f'SMOTE 后训练集：{X_train_res.shape}，窃电比例：{y_train_res.mean():.2%}')

# 标准化（基于训练集统计量）
scaler = StandardScaler()
X_train_res = scaler.fit_transform(X_train_res)
X_val       = scaler.transform(X_val)
X_test      = scaler.transform(X_test)

## 5. 基线模型训练

In [ ]:
# 5.1 随机森林
rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    class_weight='balanced',
    random_state=RANDOM_STATE,
    n_jobs=-1
)
rf_model.fit(X_train_res, y_train_res)
print('随机森林训练完成')

In [ ]:
# 5.2 XGBoost
# 经 SMOTE 过采样后训练集已均衡，无需设置 scale_pos_weight
xgb_model = xgb.XGBClassifier(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    use_label_encoder=False,
    eval_metric='logloss',
    random_state=RANDOM_STATE,
    n_jobs=-1
)
xgb_model.fit(
    X_train_res, y_train_res,
    eval_set=[(X_val, y_val)],
    verbose=False
)
print('XGBoost 训练完成')

## 6. 模型评估

在 **验证集** 与 **测试集** 上分别评估两个模型，指标包括：
准确率（Accuracy）、精确率（Precision）、召回率（Recall）、F1、ROC-AUC、PR-AUC。

In [ ]:
def evaluate_model(model, X_eval, y_eval, dataset_name='验证集'):
    """打印分类指标并返回字典。"""
    y_pred  = model.predict(X_eval)
    y_prob  = model.predict_proba(X_eval)[:, 1]

    metrics = {
        'Accuracy':  accuracy_score(y_eval, y_pred),
        'Precision': precision_score(y_eval, y_pred, zero_division=0),
        'Recall':    recall_score(y_eval, y_pred, zero_division=0),
        'F1':        f1_score(y_eval, y_pred, zero_division=0),
        'ROC-AUC':   roc_auc_score(y_eval, y_prob),
        'PR-AUC':    average_precision_score(y_eval, y_prob),
    }

    print(f'\n【{dataset_name}】')
    for k, v in metrics.items():
        print(f'  {k:<12}: {v:.4f}')
    return metrics, y_pred, y_prob


print('=' * 40)
print('随机森林')
rf_val_metrics,  rf_val_pred,  rf_val_prob  = evaluate_model(rf_model, X_val,  y_val,  '验证集')
rf_test_metrics, rf_test_pred, rf_test_prob = evaluate_model(rf_model, X_test, y_test, '测试集')

print('\n' + '=' * 40)
print('XGBoost')
xgb_val_metrics,  xgb_val_pred,  xgb_val_prob  = evaluate_model(xgb_model, X_val,  y_val,  '验证集')
xgb_test_metrics, xgb_test_pred, xgb_test_prob = evaluate_model(xgb_model, X_test, y_test, '测试集')

In [ ]:
# 6.1 混淆矩阵
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, pred, title in [
    (axes[0], rf_test_pred,  'Random Forest - Test Set'),
    (axes[1], xgb_test_pred, 'XGBoost - Test Set'),
]:
    cm = confusion_matrix(y_test, pred)
    disp = ConfusionMatrixDisplay(cm, display_labels=['Normal', 'Theft'])
    disp.plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(title)
plt.tight_layout()
plt.show()

In [ ]:
# 6.2 ROC 曲线
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, prob, title, auc_val in [
    (axes[0], rf_test_prob,  'Random Forest', rf_test_metrics['ROC-AUC']),
    (axes[1], xgb_test_prob, 'XGBoost',       xgb_test_metrics['ROC-AUC']),
]:
    fpr, tpr, _ = roc_curve(y_test, prob)
    ax.plot(fpr, tpr, label=f'AUC = {auc_val:.4f}', lw=2)
    ax.plot([0, 1], [0, 1], 'k--', lw=1)
    ax.set_xlabel('False Positive Rate')
    ax.set_ylabel('True Positive Rate')
    ax.set_title(f'ROC Curve - {title}')
    ax.legend(loc='lower right')

plt.tight_layout()
plt.show()

In [ ]:
# 6.3 PR 曲线
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, prob, title, pr_auc in [
    (axes[0], rf_test_prob,  'Random Forest', rf_test_metrics['PR-AUC']),
    (axes[1], xgb_test_prob, 'XGBoost',       xgb_test_metrics['PR-AUC']),
]:
    prec, rec, _ = precision_recall_curve(y_test, prob)
    ax.plot(rec, prec, label=f'PR-AUC = {pr_auc:.4f}', lw=2)
    baseline = y_test.mean()
    ax.axhline(y=baseline, color='k', linestyle='--', lw=1, label=f'Baseline = {baseline:.3f}')
    ax.set_xlabel('Recall')
    ax.set_ylabel('Precision')
    ax.set_title(f'PR Curve - {title}')
    ax.legend(loc='upper right')

plt.tight_layout()
plt.show()

In [ ]:
# 6.4 特征重要性（XGBoost）
importances = xgb_model.feature_importances_
feat_imp_df = pd.DataFrame({'feature': feat_names, 'importance': importances})
feat_imp_df = feat_imp_df.sort_values('importance', ascending=False)

plt.figure(figsize=(10, 5))
sns.barplot(data=feat_imp_df, x='importance', y='feature', palette='viridis')
plt.title('XGBoost Feature Importances')
plt.xlabel('Importance Score')
plt.ylabel('Feature')
plt.tight_layout()
plt.show()

print(feat_imp_df.to_string(index=False))

## 7. 结果汇总

In [ ]:
summary = pd.DataFrame([
    {'模型': '随机森林',  '数据集': '验证集', **rf_val_metrics},
    {'模型': '随机森林',  '数据集': '测试集', **rf_test_metrics},
    {'模型': 'XGBoost',  '数据集': '验证集', **xgb_val_metrics},
    {'模型': 'XGBoost',  '数据集': '测试集', **xgb_test_metrics},
])
summary = summary.set_index(['模型', '数据集'])
summary = summary.round(4)
print(summary.to_string())
summary

---

## 后续改进方向

- **更丰富的特征**：时频分析（FFT 系数）、滑动窗口统计、节假日/工作日分离等
- **深度学习**：LSTM / Transformer 直接建模时序数据
- **超参数调优**：Optuna / GridSearchCV
- **模型融合**：Stacking / Voting
- **可解释性**：SHAP 值分析